In [ ]:
# imports

import os
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama

In [ ]:
# Constantes

OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL = "llama3.2"

Flujo completo:
 → Website()          # descarga y extrae el texto
 → user_prompt_for()  # construye el mensaje con ese texto
 → messages_for()     # empaqueta system_prompt + user_prompt
 → ollama.chat()      # envía al modelo
 → respuesta en Markdown en español

In [ ]:
# Una clase para representar una página web

class Website:
    """
    Una clase de utilidad para representar un sitio web que hemos scrappeado
    """

    def __init__(self, url):
        """
        Crea este objeto de sitio web a partir de la URL indicada utilizando la biblioteca BeautifulSoup
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No tiene título"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

Comprobamos que la clase Website y el scraping funciona

In [ ]:
frog = Website("https://cursos.frogamesformacion.com")
print(frog.title)
print(frog.text)

In [ ]:
# Define nuestro mensaje de sistema: puedes experimentar con esto más tarde, cambiando la última oración a "Responder en Markdown en español".
# Es la instrucción inicial que recibe el modelo antes de que empiece a trabajar. Le dice quién es y cómo debe comportarse.
# Sin este prompt, el modelo respondería en inglés y con un formato genérico. Con él, sabe que debe resumir, ignorar navegación y responder en español con formato Markdown.

system_prompt = "Eres un asistente que analiza el contenido de un sitio web \
y proporciona un breve resumen, ignorando el texto que podría estar relacionado con la navegación. \
Responder en Markdown en español."

In [ ]:
# Construye el mensaje con el contenido real que se envía el modelo en cada llamada. Combina dos cosas, una instrucción "resume este sitio web" y el contenido real extraído por la clase Website:

def user_prompt_for(website):
    user_prompt = f"Estás viendo un sitio web titulado {website.title}"
    user_prompt += "\nEl contenido de este sitio web es el siguiente; \
    proporciona un breve resumen de este sitio web en formato Markdown. \
    Si incluye noticias, productos o anuncios, resúmelos también.\n\n"
    user_prompt += website.text
    return user_prompt

In [ ]:
# Verificar que el prompt tiene buena pinta antes de hacer la llamada
print(user_prompt_for(frog))

In [ ]:
# Empaqueta los dos prompts en el formato que espera la API del modelo.
# Los modelos como Ollama o OpenAI no reciben el texto directamente — esperan una lista de mensajes con una estructura específica donde cada mensaje tiene un rol y un contenido.

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [ ]:
# Y ahora: llama a la API de Ollama

def summarize(url):
    website = Website(url)
    payload = {
        "model": MODEL,
        "messages": messages_for(website),  # ← incluye system + user
        "stream": False
    }
    response = requests.post(OLLAMA_API, json=payload, headers=HEADERS)
    message = response.json()['message']['content']
    return message

In [ ]:
# Comento para no ejecutar esta instrucción cuando después vuelve a ejecutarse con formato Markdown
# summarize("https://cursos.frogamesformacion.com")

In [ ]:
# Una función para mostrar esto de forma clara en la salida de Jupyter, usando markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://cursos.frogamesformacion.com")

In [ ]:
# Forma 2 - Usando la librería oficial de Ollama (más limpia)

def summarize_v2(url):
    website = Website(url)
    response = ollama.chat(
        model=MODEL,
        messages=messages_for(website)
    )
    return response['message']['content']

In [ ]:
def display_summary_v2(url):
    summary = summarize_v2(url)
    display(Markdown(summary))

In [17]:
display_summary_v2("https://cursos.frogamesformacion.com")

KeyboardInterrupt: 